### np.argmax

In [3]:
def get_shape(a):
    if not isinstance(a, list):
        return ()
    if len(a) == 0:
        return (0,)
    inner_shape = get_shape(a[0])
    for i in range(1, len(a)):
        if get_shape(a[i]) != inner_shape:
            raise ValueError(
                f"Jagged array: first row shape {inner_shape}, "
                f"but row {i} shape {get_shape(a[i])}"
            )
    return (len(a),) + inner_shape


def flatten(a):
    if not isinstance(a, list):
        return [a]
    result = []
    for item in a:
        result.extend(flatten(item))
    return result


def _product(shape):
    p = 1
    for s in shape:
        p *= s
    return p


def reshape(flat, shape):
    if len(shape) == 0:
        return flat[0]
    if len(shape) == 1:
        return flat[:]
    chunk = _product(shape[1:])
    return [reshape(flat[i * chunk:(i + 1) * chunk], shape[1:]) for i in range(shape[0])]


def argmax_1d(lst):
    if len(lst) == 0:
        raise ValueError("attempt to get argmax of an empty sequence")
    max_idx = 0
    for i in range(1, len(lst)):
        if lst[i] > lst[max_idx]:
            max_idx = i
    return max_idx


def argmax_along_axis(a, axis, shape):
    ndim = len(shape)

    out_shape = shape[:axis] + shape[axis + 1:]

    if len(out_shape) == 0:
       
        flat = flatten(a)
        return argmax_1d(flat)

    total_out = _product(out_shape)
    results = []

    for out_flat_idx in range(total_out):
       
        out_coords = []
        tmp = out_flat_idx
        for dim in reversed(out_shape):
            out_coords.append(tmp % dim)
            tmp //= dim
        out_coords.reverse()

        slice_len = shape[axis]
        slice_vals = []
        for k in range(slice_len):
           
            full_coords = out_coords[:axis] + [k] + out_coords[axis:]
            elem = a
            for c in full_coords:
                elem = elem[c]
            slice_vals.append(elem)

        results.append(argmax_1d(slice_vals))

    return reshape(results, out_shape)


def np_argmax(a, axis=None):
   
    shape = get_shape(a)
    ndim  = len(shape)

    if ndim == 0:
        raise ValueError("attempt to get argmax of a 0-d array")

    if axis is None:
        flat = flatten(a)
        if len(flat) == 0:
            raise ValueError("attempt to get argmax of an empty sequence")
        return argmax_1d(flat)

    if axis < 0:
        axis += ndim
    if axis < 0 or axis >= ndim:
        raise ValueError(f"axis {axis} is out of bounds for array of dimension {ndim}")

    return argmax_along_axis(a, axis, shape)

### test cases 

In [4]:
print(np_argmax([3, 1, 4, 1, 5, 9, 2, 6]))
print(np_argmax([[10, 17, 25], [15, 11, 22]]))
print(np_argmax([[10, 17, 25], [15, 11, 22]], axis=0))
print(np_argmax([[10, 17, 25], [15, 11, 22]], axis=1))
print(np_argmax([5, 5, 5]))
print(np_argmax([[5, 5], [5, 5]], axis=0))
print(np_argmax([[5, 5], [5, 5]], axis=1))
print(np_argmax([[[1, 9], [3, 4]], [[5, 2], [8, 0]]], axis=0))
print(np_argmax([[[1, 9], [3, 4]], [[5, 2], [8, 0]]], axis=1))
print(np_argmax([[[1, 9], [3, 4]], [[5, 2], [8, 0]]], axis=2))
print(np_argmax([42]))
print(np_argmax([[7]]))
print(np_argmax([3, 1, 4, 1, 5, 9, 2, 6], axis=0))
print(np_argmax([3, 1, 4, 1, 5, 9, 2, 6], axis=-1))

5
2
[1, 0, 0]
[2, 2]
0
[0, 0]
[0, 0]
[[1, 0], [1, 0]]
[[1, 0], [1, 0]]
[[1, 1], [0, 0]]
0
0
5
5


In [5]:
print(np_argmax([]))

ValueError: attempt to get argmax of an empty sequence

In [6]:
print(np_argmax([[10, 17, 25], [15, 11, 22]], axis=2))

ValueError: axis 2 is out of bounds for array of dimension 2

In [7]:
print(np_argmax([[10, 17, 25], [15, 11, 22]], axis=-3))

ValueError: axis -1 is out of bounds for array of dimension 2

In [8]:
print(np_argmax([[1, 2], [3]]))

ValueError: Jagged array: first row shape (2,), but row 1 shape (1,)

In [9]:
print(np_argmax(42))

ValueError: attempt to get argmax of a 0-d array